# 00 · Orientation

**What this notebook is for.** A tour of the repository, done by *using* it: you will draw
sparse features, build a toy model, train it, measure it, and read the record an experiment
run leaves behind. Everything here calls the library in `src/superposition/`; nothing is
re-implemented.

**Learning goals**

1. Know where each piece lives: data, models, training, metrics, experiments, runs, figures.
2. Run the smallest end-to-end loop by hand: data → model → loss → metrics.
3. Read an experiment run: its config, manifest and acceptance results.

**Paper pages:** none yet; this is the map. Notebook 01 starts the paper (pp. 1–8).

> **How to use these notebooks.** Before every code cell marked **Predict**, write down what
> you expect. Then run it. The gap between the two is where you learn.

In [ ]:
# Runs locally and in Colab. In a fresh environment (e.g. Colab), install the package first.
# Replace <OWNER> with the GitHub owner of your copy of the repository.
try:
    import superposition  # noqa: F401
except ImportError:
    %pip install -q "git+https://github.com/<OWNER>/superposition.git"

## 1. The layout

| Where | What | Paper |
|---|---|---|
| `superposition.data` | sparse feature distributions (`IndependentSparse`, correlated / anticorrelated sets) | p. 9, p. 23 |
| `superposition.models` | `TiedModel` (linear, ReLU output, ReLU hidden) and `AbsModel`, all with a leading *instance* dimension | pp. 10, 31, 35 |
| `superposition.train` | one vectorized trainer for many independent models at once | p. 10 |
| `superposition.metrics` | ‖Wᵢ‖, superposition score, Dᵢ, D*, geometry graphs … (pure functions) | pp. 8–20 |
| `superposition.experiments` | one module per paper experiment (E00, E01, …) with acceptance criteria | §7 of SPEC.md |
| `superposition.runs` | run directories under `results/`, manifests, checkpoints | — |
| `superposition.plotting` | paper-style figures | — |

The **instance dimension** is the one idea to absorb first: every parameter has shape
`(inst, …)`, and instance k is a completely independent model. That is how a whole sparsity
sweep, with several restarts each, trains as a single PyTorch model.

In [ ]:
from superposition.experiments import REGISTRY

for name, exp in REGISTRY.items():
    print(f"{exp.code}  {name:<16} {exp.paper_pages:<10} {exp.title}")

## 2. Configs and profiles

Each experiment has `configs/<name>.yaml` with a `base` block and two profiles: `quick` (≤ 10
minutes on a laptop CPU) and `full` (paper scale). They load into typed dataclasses.

**Predict:** E00 trains the ReLU output model at three sparsities with 8 restarts each. How many
independent models is that, and how many instances does the ReLU model have?

In [ ]:
from superposition.experiments.runner import load_experiment_config

config = load_experiment_config("intro_pentagon", "quick")
print(config.sparsities, config.n_restarts, config.train)
print("instances per model type:", len(config.sparsities) * config.n_restarts)

## 3. Data: sparse features (p. 9)

Each feature xᵢ is 0 with probability S (the *sparsity*) and otherwise uniform on [0, 1]. The
code uses the **feature probability** p = 1 − S.

**Predict:** for two instances with p = 1 and p = 0.1, what fraction of entries is nonzero, and
what is the mean of the nonzero values?

In [ ]:
import torch

from superposition.data import IndependentSparse, geometric_importance
from superposition.utils import make_generator

dist = IndependentSparse(feature_probability=torch.tensor([1.0, 0.1]), n_inst=2, n_features=5)
x = dist.sample(10_000, make_generator(0))
print("shape (batch, inst, feat):", tuple(x.shape))
print("fraction nonzero per instance:", (x != 0).float().mean(dim=(0, 2)).tolist())
print("mean of nonzero values:", round(x[x != 0].mean().item(), 3))

## 4. A model (p. 10)

The ReLU output model is x′ = ReLU(WᵀWx + b). `W` has shape `(inst, hidden, feat)`: column
`W[k, :, i]` is feature i's direction in instance k.

**Predict:** with n = 5 features, m = 2 hidden dimensions and 2 instances, what are the shapes
of `W`, `b` and `model(x)`?

In [ ]:
from superposition.models import TiedModel

model = TiedModel.from_kind("relu_output", n_features=5, n_hidden=2, n_inst=2, seed=0)
print("W:", tuple(model.W.shape), " b:", tuple(model.b.shape), " x′:", tuple(model(x).shape))

## 5. Train it

The loss is Σᵢ Iᵢ (xᵢ − x′ᵢ)², with importance Iᵢ = 0.9ⁱ here. It is summed over features,
averaged over the batch (one number per instance), then summed over instances for
`backward()`, so the two instances never influence each other.

**Predict:** which instance will end with the lower loss, the dense one (p = 1) or the sparse
one (p = 0.1)? Does that mean the sparse model is "better"? (Footnote 17, p. 56.)

In [ ]:
from superposition.train import train

importance = geometric_importance(5, 0.9)
result = train(model, dist, importance, steps=2_000, eval_samples=20_000)
print("final loss per instance:", [round(v, 4) for v in result.final_loss.tolist()])

The sparse instance has a much lower loss, but only because its task is easier: most of its
inputs are zero. Losses at different sparsities measure different tasks and are **not
comparable** (footnote 17).

## 6. Measure it (p. 11)

Two numbers per feature carry most of the story: the norm ‖Wᵢ‖ (≈ 1 if represented, ≈ 0 if
not) and the superposition score Σ_{j≠i}(Ŵᵢ·Wⱼ)² (0 if orthogonal to everything else).

In [ ]:
from superposition import metrics

W = model.W.detach()
print("‖Wᵢ‖:\n", metrics.feature_norms(W).round(decimals=2))
print("superposition score:\n", metrics.superposition_score(W).round(decimals=2))
print("dimensions per feature D*:", metrics.dims_per_feature(W).round(decimals=2).tolist())

After only 2,000 steps these models are not finished; E00 trains for 10,000 steps with 8
restarts. Notebook 02 does it properly.

## 7. How a run is recorded

`superposition run <experiment>` (or `run_experiment` from Python) writes
`results/<experiment>/<run_id>/` with the resolved `config.yaml`, a `manifest.json` (git commit,
package versions, device, seed, timings), `metrics.parquet`, `acceptance.json`, checkpoints,
figures and a log. Every number in `docs/REPRODUCTION.md` points to one of these.

In [ ]:
from superposition.runs import latest_run

run = latest_run("intro_pentagon")
if run is None:
    print("No E00 run yet. Create one with:  uv run superposition run intro_pentagon")
else:
    print(run.path)
    for path in sorted(run.path.rglob("*")):
        print("  ", path.relative_to(run.path))
    doc = run.read_acceptance()
    print("overall:", doc["overall"], f"({doc['n_passed']}/{doc['n_graded']} criteria)")

## Exercises

**1.** Build `TiedModel.from_kind("linear", 5, 2, n_inst=2, seed=0)` and
`TiedModel.from_kind("linear", 5, 2, n_inst=4, seed=0)`. Are the first two instances' weights
identical? Why does the repository insist on this?

<details><summary>Solution</summary>

```python
from superposition.models import TiedModel
a = TiedModel.from_kind("linear", 5, 2, n_inst=2, seed=0).W
b = TiedModel.from_kind("linear", 5, 2, n_inst=4, seed=0).W
print(torch.equal(a, b[:2]))  # True
```

Instance k's initialization depends only on (seed, k) (`superposition.utils.derive_seed`).
So adding restarts or sparsity levels to an experiment never changes the models that were
already there, and a result can be reproduced instance by instance.
</details>

**2.** Find the line in `src/superposition/train.py` that sums the loss over instances. Why
a sum and not a mean, and why does Adam not care which?

<details><summary>Solution</summary>

`objective.sum().backward()`. With a sum, each instance's gradient is exactly what it would be
if it were trained alone. A mean would divide every gradient by the number of instances; Adam
divides each update by a running estimate of the gradient's scale, so a constant factor cancels
(up to Adam's ε). The sum is simply the clearer statement of independence.
</details>

**3.** Open a run's `manifest.json` (run E00 first if you have none). Which fields would you
need to reproduce the run exactly?

<details><summary>Solution</summary>

`git.commit` (and `git.dirty` must be false), `seed`, `device`, the package versions, and the
resolved `config.yaml` next to it. Re-running `superposition run intro_pentagon --seed <seed>
--device <device>` at that commit reproduces it (bit-for-bit on the same hardware and
software).
</details>

**4.** Sample from `IndependentSparse(0.2, n_inst=1, n_features=5, low=-1.0, high=1.0)`. What
changes, and which experiment needs it? (Hint: p. 35.)

<details><summary>Solution</summary>

Nonzero values become uniform on [−1, 1] with mean 0. The absolute-value task (E08, p. 35)
needs signed inputs, because |x| = ReLU(x) + ReLU(−x) only needs two neurons when x can be
negative.
</details>